### The Pizza Ordering Process  

The process begins when a customer places an order. The pizzeria, operated by a single person, receives the order, prepares the pizza, and delivers it to the customer.  

The process ends when the customer receives the order. Alternatively, the customer may cancel the order, but only before it has been accepted by the pizzeria.  

Due to unforeseen circumstances, the pizzeria may cancel an order at any time. 

Each step in this process is recorded as an event in the system logs. A single order is represented as one trace, which includes multiple events along with their corresponding timestamps.  

The recorded events are:  
- **Order received** – when the customer places an order  
- **Order canceled** – when the customer cancels the order (only possible before confirmation)  
- **Order confirmed** – when the pizzeria accepts the order  
- **Order prepared** – when the pizzeria starts baking the pizza  
- **Order on the way** – when the pizzeria begins delivery  
- **Order delivered** – when the customer receives the order  

Each event is logged with its corresponding timestamp.  


### Process Simulation with simPy (discrete event simulation)

In [18]:
import simpy
import random
import statistics

In [ ]:
import simpy
import random
import datetime
import pandas as pd

wait_times = []  # Store order processing times for analysis
event_log_df = pd.DataFrame(columns=['order_id', 'activity', 'timestamp'])

class Pizzeria():
    def __init__(self, env, num_workers, start_time):
        self.env = env
        self.worker = simpy.Resource(env, num_workers)
        self.start_time = start_time
    def get_real_time(self):
        return self.start_time + datetime.timedelta(minutes=self.env.now)
    def log_activity(self, order, activity):
        real_time = self.get_real_time()
        event_log_df.loc[len(event_log_df)] = {"order_id": order, "activity": activity, "timestamp": real_time}

    def confirm_order(self, order):
        yield self.env.timeout(random.uniform(3/60, 30/60))  # Convert seconds to minutes
        #real_time = self.get_real_time()
        #print(f"Order {order} confirmed at {self.env.now:.2f} minutes")
        self.log_activity(order=order, activity="confirm order")

    def prepare_order(self, order):
        yield self.env.timeout(random.uniform(15, 45))
        #real_time = self.get_real_time()
        #print(f"Order {order} prepared at {self.env.now:.2f} minutes")
        self.log_activity(order=order, activity="prepare order")

    def deliver_order(self, order):
        yield self.env.timeout(random.uniform(10, 60))
        #real_time = self.get_real_time()
        #print(f"Order {order} delivered at {self.env.now:.2f} minutes")
        self.log_activity(order=order, activity="deliver order")

def place_order(env, order, pizzeria):
    arrival_time = env.now
    #real_arrival_time = pizzeria.get_real_time()
    #print(f"Order {order} placed at {arrival_time:.2f} minutes")
    #print(f"Order {order} placed at {real_arrival_time.strftime('%Y-%m-%d %H:%M:%S')}")
    pizzeria.log_activity(order, "place order")
    
    # Order confirmation
    with pizzeria.worker.request() as request:
        yield request
        yield env.process(pizzeria.confirm_order(order))
    
    # Order preparation
    with pizzeria.worker.request() as request:
        yield request
        yield env.process(pizzeria.prepare_order(order))
    
    # Order delivery
    with pizzeria.worker.request() as request:
        yield request
        yield env.process(pizzeria.deliver_order(order))
    
    completion_time = env.now
    real_completion_time = pizzeria.get_real_time()
    wait_times.append(completion_time - arrival_time)
    print(f"Order {order} completed at {real_completion_time.strftime('%Y-%m-%d %H:%M:%S')} minutes (Total wait time: {completion_time - arrival_time:.2f} minutes)")

def run_pizzeria(env, num_workers, start_time):
    pizzeria = Pizzeria(env=env, num_workers=num_workers, start_time=start_time)
    order = 0
    
    # Generate initial orders
    for i in range(3):
        env.process(place_order(env, order, pizzeria))
        order += 1
    
    # Generate orders over time
    while True:
        yield env.timeout(random.uniform(3, 5))  # New order every 3-5 minutes
        env.process(place_order(env, order, pizzeria))
        order += 1

# Simulation parameters
start_time = datetime.datetime.now()
env = simpy.Environment()
env.process(run_pizzeria(env, num_workers=1, start_time=start_time))
env.run(until=8*60)  # Run for 10 minutes


Order 0 completed at 2025-04-02 15:07:42 minutes (Total wait time: 119.03 minutes)
Order 1 completed at 2025-04-02 16:07:15 minutes (Total wait time: 178.57 minutes)
Order 2 completed at 2025-04-02 16:40:56 minutes (Total wait time: 212.25 minutes)


In [11]:
event_log_df

,order_id,activity,timestamp
0,0,place order,2025-04-02 13:08:41.393394
1,1,place order,2025-04-02 13:08:41.393394
2,2,place order,2025-04-02 13:08:41.393394
3,0,confirm order,2025-04-02 13:09:06.880475
4,1,confirm order,2025-04-02 13:09:12.006818
...,...,...,...
166,119,place order,2025-04-02 20:55:54.102333
167,120,place order,2025-04-02 20:59:40.048402
168,121,place order,2025-04-02 21:02:53.524132
169,10,prepare order,2025-04-02 21:04:13.382572


In [ ]:
sorted_df = event_log_df.sort_values(by='timestamp')
sorted_df.iloc[:50]

### Problems:
- slow processing time, which is not realistic
- what is gonna happen to all of these placed order, which are not processed
- Or I can try with bizagi, which also use DES
- Agent need its state 

In [15]:
df = event_log_df[event_log_df['order_id']==0]
df

,order_id,activity,timestamp
0,0,place order,2025-04-02 13:08:41.393394
3,0,confirm order,2025-04-02 13:09:06.880475
12,0,prepare order,2025-04-02 13:33:20.532581
46,0,deliver order,2025-04-02 15:07:42.971277


In [16]:
df = event_log_df[event_log_df['order_id']==2]
df

,order_id,activity,timestamp
2,2,place order,2025-04-02 13:08:41.393394
5,2,confirm order,2025-04-02 13:09:34.179347
30,2,prepare order,2025-04-02 14:32:48.224335
87,2,deliver order,2025-04-02 16:40:56.438446


In [17]:
df = event_log_df[event_log_df['order_id']==3]
df

,order_id,activity,timestamp
6,3,place order,2025-04-02 13:12:14.807877
31,3,confirm order,2025-04-02 14:33:04.400641
98,3,prepare order,2025-04-02 17:21:44.857361


In [ ]:
df = event_log_df[event_log_df['order_id']==0]

### What can be controlled and what not?
### What information/action each agent can have access to/act?

### What can happen in reality:

- **Order Overload**: A high volume of orders arriving simultaneously or within a short timeframe can overwhelm the pizzeria, causing delays.
- **Processing Errors**: The pizzeria may make mistakes, e.g. baking the wrong pizza, missing ingredients, leading to additional preparation time.
- **External Factors**: Weather conditions, traffic problems, or the distance to the customer can impact delivery times,  equipment failures, e.g., a broken oven, can slow down preparation or even result in order cancellations.
- **Concurrent Processing**: Orders follow a sequence but can overlap in execution, e.g. new order may arrive while the pizzeria is already preparing or delivering another.
- **Multitask**: The pizzeria can multitask, e.g. taking a new order while baking a pizza or delivering another order, or they can be **lazy**

### What we need to simulate:

- **Order Arrival Variability**: Orders may follow a predictable pattern, e.g. more orders during lunch time, but also may exhibit some randomness, e.g. due to the weather or other reasons.
- **Preparing Time Variability**: Different preparation times due to workload, mistakes, or ingredient availability.
- **Environmental Factors**: Weather, traffic, and customer distance should dynamically affect delivery times.
- **Multitasking Capabilities**: The simulation should allow the pizzeria to handle multiple tasks simultaneously, e.g. taking orders while baking or delivering.
- **Independent Agent Behavior**: Agents take actions independently on other agents, simulating concurrent processing.

### Note: When it is real, we do not need a lot of event logs.

### Simulation with LLM-Agents

**Customer Agent**
- number: 10
- profile:
    + Name
    + Charater
- state:
    + Where do they live, e.g., how far do they live from the pizzeria
- actions:
    + place an order 
    + cancel order (depending on **Pizerria Agent**)
    + receive order (depending on **Pizzeria Agent**)
- access to:
    + order id
    + ordering time
    + order states


**Pizzeria Agent**
- number: 1
- profile:
    + Name
    + Character
    + Multi-tasking skills
    + Working time/style
    + Mistakes: preparing order before confirming, baking wrong pizza
- actions:
    + confirm order: how long does it take, what can cause delays?
    * prepareing order: like above
    * deliver: like above
    * do nothing
- access to:
    + placed order
    + ordering time
    + delivery place
    + order states (e.g. there is one placed order waiting for confirmation, pizza baked)
    + cancel order (rare)
    + do nothing
**Global states**:
- weather
- traffic (where, when)
- process problems: not enough ingredients (rare), machine mal-functions (rare)
**Strategies**:
- Goal: agent takes one of actions at a specific time, which will change the process state and other agents's actions
- What decide **Customer Agent**'s action:
    + weather
    + character, e.g. undecisive therefore want to cancel order
    + how far they are from pizzeria
    + randomness
    + order stated
    + **Pizzeria Agent**'s action: if he/she is gonna deviler or not to receive order
- What decide **Pizzeria Agent**'s action:
    + other agents' action, order placed or not
    + working style: multitasking or lazy
    + logical reasoning, e.g. can preparing 2,3 pizza at the same time
    + global factors can affect activity time


In [ ]:
# customer agent
# inputs: name, character, distance from pizzeria, weather, traffic, order_state (if already placed)
# output: action, timestamp, order_id, pizza_time, if action is do nothing, then output nothing?
def create_customer_agent(name, character, distance):
    pass
# pizzeria agent
# input: work_profile, states of all incompleted order (how to input these information during simulation and not at the beginning), waiting mode or not (while waiting for baking or delivering)
# output: action, timestamp (starting-ending)
# output: if something is done
# so either you need an enviroment to interact or the llm understand so express it in timestamp.
def create_pizzeria_agent(work_profile):
    pass

Possible approaches:
- llm interact with environment
- llm work under a supervisor, who acts like the ordering application, which can stay update with all information
    + call on random customer
    + inform pizzeria if new order or any change in order state.
    + update new information everytime a agent doing something ()
    + through prompts
    + state-aware
    + solve pronlem of real time or not
    + send message/feed back and other agents decide what to do, for example, your order is placed, on order is placed, agent carry out action and feedback back to supervisor, supervisor update information
- Question: 
    + How to store agent's profile? in prompt?
    + How to keep track of realtime to determine what the information the supervisor can have? (thorough in start_time)
    + Should I generate the orders all at once at the beginning (which i can decide the number of orders), or decide a period and let supervisor pick random customers
    + Realtime information should always give to each agent. Now is datetime, do you want to order a pizza or not (pizza, time corresponding to.)
    + What about changing prompts: uniform one + feedback from system, in this case (supervisor)

    + Don't put timestamp in prompt, but if realtime it will take forever

In [ ]:
### Helper function
from datetime import datetime
import random
import string

# generating random order ID
def generate_random_id():
    # Generate 3 random letters
    letters = ''.join(random.choices(string.ascii_uppercase, k=3))   
    # Generate 3 random digits
    digits = ''.join(random.choices(string.digits, k=3))
    # Combine the letters and digits
    random_id = letters + digits
    return random_id
def get_weather(timestamp):
    # weather 
    # - 9:00-13:00 rainy
    # - 13:00 - 17:00: calm 
    # - 17:00-22:00: very windy
    """dt = datetime.fromisoformat(timestamp)
    hour = dt.hour"""
    hour = timestamp.hour
    if hour >= 9 and hour <=13:
        return 'rainy'
    elif hour >13 and hour <=17:
        return 'calm'
    elif hour > 17 and hour < 22:
        return 'very windy'
    else:
        return 'No weather available for the given hour'
def get_traffic(timestamp, distance):
    """dt = datetime.fromisoformat(timestamp)
    hour = dt.hour"""
    hour = timestamp.hour
    if distance <=6:
        if (hour >=12 and hour <=13) or (hour >=17 and hour <=20):
            return 'busy traffic'
        else:
            return 'normal traffic'
    else:
        return 'normal traffic'

# generate randon customer profile
from prompts import characters, names
used_combinations = set()
used_characters = set()

def generate_unique_order_data():
    if len(used_characters) >= len(characters):
        raise ValueError("All characters have been used. Cannot generate more unique orders.")

    name = random.choice(names)
    distance = str(random.randint(1, 20))  # Distance in km
    available_characters = list(set(characters) - used_characters)
    
    if not available_characters:
        raise ValueError("No more unique characters available.")
    
    character = random.choice(available_characters)

    combination = (name, distance, character)
    used_combinations.add(combination)
    used_characters.add(character)
    
    return combination

# Generating random possible numper of order basing on time
import numpy as np
import datetime

def generate_num_orders(timestamp):
    """
    Generate the number of orders arriving at a pizzeria based on the time of day 
    using a Poisson distribution.

    Parameters:
        timestamp (datetime.datetime): The current timestamp.

    Returns:
        int: Number of orders at this timestamp.
    """
    hour = timestamp.hour

    # Define the average number of orders per minute (lambda) based on time of day
    arrival_rates = {
        11: 2, 12: 4, 13: 3, 14: 2,  # Lunch peak
        18: 3, 19: 5, 20: 4, 21: 3,  # Dinner peak
        8: 0.5, 9: 1, 10: 1.5,       # Morning low
        15: 1, 16: 1.5, 17: 2,       # Afternoon moderate
        22: 1.5, 23: 0.5,            # Late night low
    }
    
    # Default to a very low arrival rate for early morning (0:00 - 7:59) or other unspecified hours
    default_lambda = 0.2 if hour < 8 or hour > 23 else 1

    # Get arrival rate (λ) for the given hour
    lambda_val = arrival_rates.get(hour, default_lambda)

    # Generate number of orders from Poisson distribution
    num_orders = np.random.poisson(lambda_val)

    return num_orders

# get latest sate of an order 
def get_latest_states(system_state) -> str:
    output_strings = []
    
    for order_id, pizza_type, state_list, timestamp_list in zip(
        system_state['order_ids'], 
        system_state['pizza_types'], 
        system_state['states'], 
        system_state['timestamp']
    ):
        if state_list and timestamp_list:  # Ensure lists are not empty
            latest_state = state_list[-1]
            latest_timestamp = timestamp_list[-1]
            output_strings.append(f"Order id {order_id} ({pizza_type}) is {latest_state} at {latest_timestamp}")
    
    return ", ".join(output_strings)

In [38]:
from dotenv import load_dotenv
import os

load_dotenv()

api_key = os.getenv('GOOGLE_API_KEY')

from langchain_google_genai import ChatGoogleGenerativeAI

llm = ChatGoogleGenerativeAI(model='gemini-2.0-flash', temperature=1.0)

In [ ]:
pizzeria_menu = """
    --- Pizzeria Menu ---
    Regular Menu:
    - Margherita: $8.50
    - Pepperoni: $9.50
    - Vegetarian: $9.00
    - BBQ Chicken: $10.00
    
    Special Lunch Menu (Available everyday, 11:00 - 14:00):
    - Truffle Mushroom Pizza: $7.50
    - Spicy Salami Pizza: $7.00
    - Pesto Chicken Pizza: $7.50
    """

In [ ]:
# pydantic to force structured output (log event) produced by llm agent
from pydantic import BaseModel, Field
from langchain_core.prompts import PromptTemplate

class CustomerOrderOutput(BaseModel):
    """Structured Output of Customer Order"""
    action: str = Field(description="Action taken by customer")
    pizza_type: str = Field(description="The kind of pizza the customer want to order, leave empty if customer does not order")
    timestamp: str = Field(description='The timestamp at which the activity is finished')

class CustomerCancelOutput(BaseModel):
    """Structured Output of Customer Order"""
    action: str = Field(description="Action taken by customer")
    pizza_type: str = Field(description="The kind of pizza the customer want to order, leave empty if customer does not order")
    timestamp: str = Field(description='The timestamp at which the activity is finished')


def create_order_agent(name, distance, character, current_time):
    if not character:
        character = "You usually eat healthy, but occationally you crave pizza for lunch and you prefer not to wait too long."
    customer_order_prompt = f"""
    You are {name}, living {distance} km from Da Pipo Pizzeria. 
    {character}

    Do you want to order a pizza from Da Pipo Pizzeria?

    Here are the menu: 
    {pizzeria_menu}

    Your decision to order depends on multiple factors:  
    - **Weather** (e.g., rainy weather makes it a great time for pizza but may delay delivery)  
    - **Time** (e.g., you're lazier to cook on weekends)  
    - **Hunger level** (whether you're hungry right now)  
    - **Traffic** (heavy traffic may slow delivery)  
    - **Distance** (closer distance means faster delivery)  

    Considering these factors, decide whether to order pizza or not. Respond concisely with:  
    - **action**: `"place order"` or `"not place order"`  
    - **pizza type**: The type of pizza you want (if ordering), otherwise leave it empty.
    - **timestamp**: The time at which you finalize your decision (current time: {current_time})  
    """
    prompt = PromptTemplate.from_template(customer_order_agent)
    structured_llm = llm.with_structured_output(CustomerOrderOutput)
    customer_order_agent = prompt| structured_llm
    return customer_order_agent

def create_cancel_agent(name, distance, character, pizza_type, timestamp, current_time):

    customer_action_prompt = f"""
    You are {name}, living {distance} km from Da Pipo Pizzeria. 
    {character}

    You just placed an order for **{pizza_type}** at **{timestamp}**. The order is registered but not yet confirmed.  

    You should only **cancel the order** for special reasons, such as:  
    - **A sudden change in plans** (e.g., you need to leave home unexpectedly)    
    - **A mistake in your order** (e.g., you ordered the wrong pizza type)  

    Otherwise, you should wait for your order.  

    Now, decide your action and provide the following output:  

    - **action**: `"cancel order"` or `"wait for order"` 
    - **timestamp**: The time at which you finalize your decision (current time: {current_time})   

    Please choose your action.
    """
    prompt = PromptTemplate.from_template(customer_action_prompt)
    structured_llm = llm.with_structured_output(CustomerOrderOutput)
    customer_cancel_agent = prompt| structured_llm
    return customer_cancel_agent

In [ ]:
# system state
from langchain_core.prompts import PromptTemplate
from typing import Literal
from langgraph.types import Command
from langgraph.graph import StateGraph, MessagesState, START, END

from typing import List, Dict
from typing_extensions import TypedDict

class OrderState(TypedDict):
    order_id: str
    pizza_type: str
    states: List[str]
    timestamps: List[str]

class SystemState(TypedDict):
    order_ids: List[str]
    pizza_types: List[str]
    states: List[List[str]]
    timestamp: List[List[str]]
    messages: List[str]
    current_times: List[str]
    

##### Pizzeria Agent

In [ ]:
from typing import List
from pydantic import BaseModel, Field

class PizzeriaOutput(BaseModel):
    """Structured Output of System Agent"""
    order_id: str = Field(description="Order ID")
    action: str = Field(description="Action taken by the process participant")
    timestamp: str = Field(description="The timestamp at which the action is taken place")

class PizzeriaOutputMultiple(BaseModel):
    multiple_actions: List[PizzeriaOutput] = Field(description="Multiple actions taken by the process participant")

def pizzeria_agent(state:SystemState)-> SystemState:
    pizzeria_prompt = """
    You are the owner and sole worker of Da Pipo Pizzeria. Your pizzeria operates exclusively through online orders.  

    When a new order is placed, you must handle all necessary steps to fulfill it efficiently. These steps include:  
    - Confirming the order: Unconfirmed orders can be canceled. This usually takes 5-30 seconds.  
    - Preparing the order: Baking a pizza, which typically takes 20 minutes.  
    - Delivering the order: Delivery time depends on weather, traffic, and the distance to the customer.  

    To optimize time, you can manage time in a efficient way and do multi-tasking, such as:  
    - Bake multiple pizzas simultaneously if you're notified of multiple orders at the same time.  
    - Confirm new orders or start baking another pizza while waiting for another one to be finished.  
    - Deliver multiple orders in one trip if they are nearby.  
    However, you cannot bake while driving. Be flexible and prioritize tasks based on order status and time efficiency.  

    Actions you can take: 
    - Confirm order
    - Start prepare order
    - Start deliver order
    - Finishing delivering order
    - Cancel order (only under special circumstances)
    - Do nothing (when you have nothing to do but wait for the pizza to be baked or you're on the way to deliver)
    You can start prepare an order right after confirming it because confirming doesn't take long. 
    However, you can't start delivering right after preparing because baking the pizza take times.

    Conditions for Canceling an Order
    Cancel an order only if necessary, such as:  
    - Order overload: Too many orders at once, making it impossible to fulfill all.  
    - Unavoidable issues: Equipment failure making baking pizza impossible or extreme weather conditions making delivery impossible.  

    Below is the latest state of all orders:
    {latest_state}

    And you also receive some message from the system:
    {message}

    The current time is {current_time}

    Your task: 
    Based on your time management, decide the next action(s) and output:  
    - **order_id**: The order being processed  
    - **action**: The action you are taking  
    - **timestamp**: The time at which you take the action 

    The timestamp do not have to be the current time, if you already finished the action before the current time.
    Similarly, if you can predict that you will finish one activity in shorttime, you also can output the timstamp in the short future.

    If handling multiple orders, output each action separately in the same format."""
    prompt = PromptTemplate.from_template()

##### System/Supervisor Agent

In [ ]:
class SystemOutput(BaseModel):
    """Structured Output of System Agent"""
    next_agent: str = Field(description="Next agent to be called")
    message: str = Field(description="Message to the agent being called")

def system_agent(state:SystemState)-> SystemState:
    latest_state = get_latest_states(state)
    system_prompt = """You are controlling an online ordering pizzeria. 
    Your duty is to keep orders' state updated and infor the participant involved in the order.
    You can always direct to the participants, inform them if any change of order state and in returns, they also can inform you with the state changing.
    You have access to the states of all orders, each contain:
    - order_id:
    - state_history:
    - current_time:
    Basing on this state history and the current_time, you can decide to check on the process state of each order. 
    You can choose to handoff to "pizzeria", "customer", or just wait for new order at the system.

    If one order is just placed, direct to "pizzeria" as soon as possible
    If the order is being prepared or delivered for a while, direct to "pizzeria" to look for update
    If the order is placed but not confirmed yet, direct to "customer"
    If order is cancelled or you don't think you need update, direct to the "system".

    Please output:
    - next_agent: "pizzeria", "customer", or "system"
    - message: you message to the directed person, in case you direct to "system" just leave the message empty.

    Below is the states of all orders: 
    {latest_state}
    To whom will you direct to and which message?
    """
    prompt = PromptTemplate.from_template(system_prompt)
    structured_llm = llm.with_structured_output(SystemOutput)
    chain = prompt | structured_llm
    output = chain.invoke({'latest_state': latest_state})
    if output.next_agent == 'customer':
        return Command(
            goto='customer',
            update = {'messages': state['messages'].append(output.message)}
        )
    elif output.next_agent == 'pizzeria':
        return Command(
            goto='pizzeria',
            update = {'messages': state['messages'].append(output.message)}
        )
    else:
        return Command(
            goto= END
        )